In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
)

print("Repeated-experiment notebook ready")

Repeated-experiment notebook ready


In [2]:
SEEDS = [11, 22, 33, 44, 55]
N_CLIENTS = 3
N_ROUNDS = 5
LOCAL_EPOCHS = 5
LEARNING_RATE = 0.01
CLASSES = np.array([0, 1])

print("Experiment settings:")
print("Seeds:", SEEDS)
print("Clients:", N_CLIENTS)
print("Rounds:", N_ROUNDS)
print("Local epochs:", LOCAL_EPOCHS)

Experiment settings:
Seeds: [11, 22, 33, 44, 55]
Clients: 3
Rounds: 5
Local epochs: 5


In [3]:
def compute_metrics(y_true, y_pred, y_prob):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
    }


def initialise_sgd_model(n_features, seed):
    model = SGDClassifier(
        loss="log_loss",
        learning_rate="constant",
        eta0=LEARNING_RATE,
        alpha=0.0001,
        max_iter=1,
        tol=None,
        random_state=seed,
        fit_intercept=True,
    )

    model.partial_fit(
        np.zeros((1, n_features)),
        np.array([0]),
        classes=CLASSES
    )
    return model


def get_sgd_parameters(model):
    return [model.coef_.copy(), model.intercept_.copy()]


def set_sgd_parameters(model, parameters):
    model.coef_ = parameters[0].copy()
    model.intercept_ = parameters[1].copy()
    model.classes_ = CLASSES
    return model


def weighted_average_parameters(client_parameters, client_sizes):
    total_size = sum(client_sizes)

    avg_coef = sum(
        params[0] * size for params, size in zip(client_parameters, client_sizes)
    ) / total_size

    avg_intercept = sum(
        params[1] * size for params, size in zip(client_parameters, client_sizes)
    ) / total_size

    return [avg_coef, avg_intercept]


print("Helper functions loaded")

Helper functions loaded


In [4]:
def load_heart_dataset():
    !pip -q install ucimlrepo
    from ucimlrepo import fetch_ucirepo

    heart = fetch_ucirepo(id=45)

    X = heart.data.features
    y_raw = heart.data.targets

    y = (y_raw.iloc[:, 0] > 0).astype(int)

    data = X.copy()
    data["target"] = y
    data = data.dropna()

    X = data.drop(columns=["target"]).astype(float)
    y = data["target"].astype(int)

    return X, y


def load_stroke_dataset():
    !pip -q install kagglehub
    import kagglehub
    import glob
    import os

    dataset_path = kagglehub.dataset_download(
        "fedesoriano/stroke-prediction-dataset"
    )

    stroke_file = glob.glob(
        os.path.join(dataset_path, "healthcare-dataset-stroke-data.csv")
    )[0]

    stroke_data = pd.read_csv(stroke_file)

    if "id" in stroke_data.columns:
        stroke_data = stroke_data.drop(columns=["id"])

    stroke_data = stroke_data.dropna().copy()

    y = stroke_data["stroke"].astype(int)
    X = stroke_data.drop(columns=["stroke"])

    X = pd.get_dummies(X, drop_first=True)
    X = X.astype(float)

    return X, y


print("Dataset-loading functions ready")

Dataset-loading functions ready


In [5]:
def run_repeated_experiment(dataset_name, X, y, use_class_weight=False):
    all_results = []

    for seed in SEEDS:
        print(f"Running {dataset_name} experiment with seed {seed}")

        # Train-test split
        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.20,
            random_state=seed,
            stratify=y
        )

        # Standardisation
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # Centralised LR
        class_weight = "balanced" if use_class_weight else None

        central_model = LogisticRegression(
            max_iter=2000,
            random_state=seed,
            class_weight=class_weight,
            solver="liblinear"
        )

        start = time.time()
        central_model.fit(X_train_scaled, y_train)
        central_time = time.time() - start

        central_pred = central_model.predict(X_test_scaled)
        central_prob = central_model.predict_proba(X_test_scaled)[:, 1]

        central_metrics = compute_metrics(y_test, central_pred, central_prob)
        central_metrics.update({
            "Dataset": dataset_name,
            "Seed": seed,
            "Method": "Centralised LR",
            "Training Time (s)": central_time,
        })

        all_results.append(central_metrics)

        # Hospital split
        X_hosp_a, X_remaining, y_hosp_a, y_remaining = train_test_split(
            X_train_scaled,
            y_train.to_numpy(),
            train_size=0.40,
            random_state=seed,
            stratify=y_train
        )

        X_hosp_b, X_hosp_c, y_hosp_b, y_hosp_c = train_test_split(
            X_remaining,
            y_remaining,
            train_size=35 / 60,
            random_state=seed,
            stratify=y_remaining
        )

        hospital_data = {
            "Hospital_A": (X_hosp_a, y_hosp_a),
            "Hospital_B": (X_hosp_b, y_hosp_b),
            "Hospital_C": (X_hosp_c, y_hosp_c),
        }

        # Class weighting for FL local training
        if use_class_weight:
            negative_count = np.sum(y_train.to_numpy() == 0)
            positive_count = np.sum(y_train.to_numpy() == 1)
            positive_weight = negative_count / positive_count
        else:
            positive_weight = 1.0

        def train_local_model(X_local, y_local, global_parameters):
            local_model = initialise_sgd_model(X_local.shape[1], seed)
            local_model = set_sgd_parameters(local_model, global_parameters)

            sample_weights = np.where(y_local == 1, positive_weight, 1.0)

            for _ in range(LOCAL_EPOCHS):
                local_model.partial_fit(
                    X_local,
                    y_local,
                    classes=CLASSES,
                    sample_weight=sample_weights
                )

            return local_model

        # FedAvg
        fed_start = time.time()

        global_model = initialise_sgd_model(X_train_scaled.shape[1], seed)
        global_parameters = get_sgd_parameters(global_model)

        for _ in range(N_ROUNDS):
            client_parameters = []
            client_sizes = []

            for _, (X_local, y_local) in hospital_data.items():
                local_model = train_local_model(
                    X_local,
                    y_local,
                    global_parameters
                )

                client_parameters.append(get_sgd_parameters(local_model))
                client_sizes.append(len(y_local))

            global_parameters = weighted_average_parameters(
                client_parameters,
                client_sizes
            )

            global_model = set_sgd_parameters(global_model, global_parameters)

        fed_time = time.time() - fed_start

        fed_pred = global_model.predict(X_test_scaled)
        fed_prob = global_model.predict_proba(X_test_scaled)[:, 1]

        fed_metrics = compute_metrics(y_test, fed_pred, fed_prob)
        fed_metrics.update({
            "Dataset": dataset_name,
            "Seed": seed,
            "Method": "Federated LR (FedAvg)",
            "Training Time (s)": fed_time,
        })

        all_results.append(fed_metrics)

    return pd.DataFrame(all_results)


print("Repeated experiment function ready")

Repeated experiment function ready


In [6]:
heart_X, heart_y = load_heart_dataset()

heart_repeated_results = run_repeated_experiment(
    dataset_name="Heart Disease",
    X=heart_X,
    y=heart_y,
    use_class_weight=False
)

display(heart_repeated_results.round(4))
print("Heart repeated experiments completed")

Running Heart Disease experiment with seed 11
Running Heart Disease experiment with seed 22
Running Heart Disease experiment with seed 33
Running Heart Disease experiment with seed 44
Running Heart Disease experiment with seed 55


,Accuracy,Precision,Recall,F1-score,Balanced Accuracy,ROC-AUC,Dataset,Seed,Method,Training Time (s)
0,0.8333,0.8750,0.7500,0.8077,0.8281,0.8560,Heart Disease,11,Centralised LR,0.0021
1,0.8333,0.8750,0.7500,0.8077,0.8281,0.8504,Heart Disease,11,Federated LR (FedAvg),0.0790
2,0.8500,0.8519,0.8214,0.8364,0.8482,0.9118,Heart Disease,22,Centralised LR,0.0016
3,0.8500,0.8519,0.8214,0.8364,0.8482,0.9174,Heart Disease,22,Federated LR (FedAvg),0.0814
4,0.8500,0.8519,0.8214,0.8364,0.8482,0.9241,Heart Disease,33,Centralised LR,0.0015
5,0.8500,0.8519,0.8214,0.8364,0.8482,0.9241,Heart Disease,33,Federated LR (FedAvg),0.0786
6,0.7667,0.7917,0.6786,0.7308,0.7612,0.8426,Heart Disease,44,Centralised LR,0.0016
7,0.7833,0.8261,0.6786,0.7451,0.7768,0.8404,Heart Disease,44,Federated LR (FedAvg),0.1042
8,0.9000,0.8929,0.8929,0.8929,0.8996,0.9554,Heart Disease,55,Centralised LR,0.0016
9,0.9167,0.8966,0.9286,0.9123,0.9174,0.9598,Heart Disease,55,Federated LR (FedAvg),0.0821


Heart repeated experiments completed


In [7]:
stroke_X, stroke_y = load_stroke_dataset()

stroke_repeated_results = run_repeated_experiment(
    dataset_name="Stroke Prediction",
    X=stroke_X,
    y=stroke_y,
    use_class_weight=True
)

display(stroke_repeated_results.round(4))
print("Stroke repeated experiments completed")

Using Colab cache for faster access to the 'stroke-prediction-dataset' dataset.
Running Stroke Prediction experiment with seed 11
Running Stroke Prediction experiment with seed 22
Running Stroke Prediction experiment with seed 33
Running Stroke Prediction experiment with seed 44
Running Stroke Prediction experiment with seed 55


,Accuracy,Precision,Recall,F1-score,Balanced Accuracy,ROC-AUC,Dataset,Seed,Method,Training Time (s)
0,0.7434,0.1168,0.7619,0.2025,0.7522,0.8217,Stroke Prediction,11,Centralised LR,0.0198
1,0.7403,0.1155,0.7619,0.2006,0.7506,0.8217,Stroke Prediction,11,Federated LR (FedAvg),0.1157
2,0.7617,0.1279,0.7857,0.2200,0.7732,0.8422,Stroke Prediction,22,Centralised LR,0.0180
3,0.7189,0.1074,0.7619,0.1882,0.7395,0.8252,Stroke Prediction,22,Federated LR (FedAvg),0.1080
4,0.7363,0.1322,0.9286,0.2315,0.8281,0.8752,Stroke Prediction,33,Centralised LR,0.0164
5,0.7261,0.1254,0.9048,0.2203,0.8114,0.8701,Stroke Prediction,33,Federated LR (FedAvg),0.1044
6,0.7495,0.1222,0.7857,0.2115,0.7668,0.8582,Stroke Prediction,44,Centralised LR,0.0173
7,0.7159,0.1165,0.8571,0.2051,0.7834,0.8514,Stroke Prediction,44,Federated LR (FedAvg),0.1256
8,0.7576,0.1260,0.7857,0.2171,0.7710,0.8254,Stroke Prediction,55,Centralised LR,0.0171
9,0.7637,0.1169,0.6905,0.2000,0.7287,0.8149,Stroke Prediction,55,Federated LR (FedAvg),0.1037


Stroke repeated experiments completed


In [8]:
all_repeated_results = pd.concat(
    [heart_repeated_results, stroke_repeated_results],
    ignore_index=True
)

metrics_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "Balanced Accuracy",
    "ROC-AUC",
    "Training Time (s)",
]

summary_mean = (
    all_repeated_results
    .groupby(["Dataset", "Method"])[metrics_columns]
    .mean()
)

summary_std = (
    all_repeated_results
    .groupby(["Dataset", "Method"])[metrics_columns]
    .std()
)

summary_mean_std = summary_mean.copy()

for metric in metrics_columns:
    summary_mean_std[metric] = (
        summary_mean[metric].map(lambda x: f"{x:.4f}")
        + " ± "
        + summary_std[metric].map(lambda x: f"{x:.4f}")
    )

summary_mean_std = summary_mean_std.reset_index()

print("Repeated-experiment summary: mean ± standard deviation")
display(summary_mean_std)

Repeated-experiment summary: mean ± standard deviation


,Dataset,Method,Accuracy,Precision,Recall,F1-score,Balanced Accuracy,ROC-AUC,Training Time (s)
0,Heart Disease,Centralised LR,0.8400 ± 0.0480,0.8526 ± 0.0382,0.7929 ± 0.0814,0.8208 ± 0.0591,0.8371 ± 0.0500,0.8980 ± 0.0474,0.0017 ± 0.0002
1,Heart Disease,Federated LR (FedAvg),0.8467 ± 0.0477,0.8603 ± 0.0267,0.8000 ± 0.0931,0.8276 ± 0.0603,0.8438 ± 0.0505,0.8984 ± 0.0511,0.0851 ± 0.0108
2,Stroke Prediction,Centralised LR,0.7497 ± 0.0103,0.1250 ± 0.0058,0.8095 ± 0.0673,0.2165 ± 0.0107,0.7783 ± 0.0290,0.8445 ± 0.0225,0.0177 ± 0.0013
3,Stroke Prediction,Federated LR (FedAvg),0.7330 ± 0.0196,0.1164 ± 0.0064,0.7952 ± 0.0852,0.2029 ± 0.0116,0.7627 ± 0.0341,0.8367 ± 0.0233,0.1115 ± 0.0092


In [9]:
heart_repeated_results.to_csv(
    "heart_repeated_results.csv",
    index=False
)

stroke_repeated_results.to_csv(
    "stroke_repeated_results.csv",
    index=False
)

all_repeated_results.to_csv(
    "all_repeated_results.csv",
    index=False
)

summary_mean_std.to_csv(
    "repeated_experiments_mean_std_summary.csv",
    index=False
)

print("Saved files:")
print("- heart_repeated_results.csv")
print("- stroke_repeated_results.csv")
print("- all_repeated_results.csv")
print("- repeated_experiments_mean_std_summary.csv")

Saved files:
- heart_repeated_results.csv
- stroke_repeated_results.csv
- all_repeated_results.csv
- repeated_experiments_mean_std_summary.csv
